# 🤖 ROTBOTS — AI Video Generator

---

Generates clips **only for AI scenes** (not archive/upload).

> **🤔** What does it cost to generate these clips?

---

In [ ]:
# SETUP
!pip install -q fal-client requests
import fal_client,json,os,time as _time,requests,threading
from pathlib import Path
from IPython.display import display,Markdown,Video,HTML
from google.colab import drive
drive.mount('/content/drive')
BASE_DIR=Path('/content/drive/MyDrive/rotbots_workshop')
print('✅ Setup')

In [ ]:
# API KEY
FAL_API_KEY=''
if not FAL_API_KEY: print('⚠️ Paste fal.ai key!')
else: os.environ['FAL_KEY']=FAL_API_KEY; print('✅ fal.ai')

In [ ]:
# SESSION
sessions=sorted([d.name for d in BASE_DIR.iterdir() if d.is_dir() and (d/'video_plan.json').exists()])
SESSION_NAME=sessions[-1] if sessions else ''
SESSION_DIR=BASE_DIR/SESSION_NAME
with open(SESSION_DIR/'video_plan.json') as f: plan=json.load(f)
VIDEOS_DIR=SESSION_DIR/'videos';VIDEOS_DIR.mkdir(exist_ok=True)
print(f'✅ {SESSION_NAME}')

In [ ]:
# PROMPTS
pf=SESSION_DIR/'prompts.json'
if pf.exists():
    with open(pf) as f: prompts=json.load(f)
    print(f'✅ {len(prompts)} AI prompts')
    for p in prompts: print(f'   Scene {p["scene"]}: {p["title"]} ({p["duration"]}s)')
else: print('❌ No prompts.json!')

In [ ]:
# MODEL
MODELS={'wan-t2v':{'endpoint':'fal-ai/wan-t2v','desc':'Wan 2.1','ds':5},'minimax':{'endpoint':'fal-ai/minimax/video-01','desc':'MiniMax','ds':6},'ltx-video':{'endpoint':'fal-ai/ltx-video/v2.1','desc':'LTX 2.1','ds':5}}
CHOSEN_MODEL='wan-t2v'
model=MODELS[CHOSEN_MODEL]
print(f'🤖 {CHOSEN_MODEL}: {model["desc"]}')

In [ ]:
# GENERATE
prog=display(HTML(''),display_id=True);generated_videos=[];t0=_time.time();_stop=False
def _timer(ph,si,done,total,ts):
    sp=['⠋','⠙','⠹','⠸','⠼','⠴','⠦','⠧','⠇','⠏'];tick=0
    while not _stop:
        el=_time.time()-ts;ce=_time.time()-si.get('cs',ts);avg=el/done[0] if done[0]>0 else 0;eta=(total-done[0]-1)*avg+max(0,avg-ce) if avg>0 else 0
        s=sp[tick%len(sp)];p=done[0]/total if total>0 else 0;fl=int(30*p)
        try:ph.update(HTML(f'<div style="background:#1a1a2e;padding:12px;border-radius:8px;color:#eaeaea;font-family:monospace;"><div style="font-size:16px;font-weight:bold;color:#e94560;">{s} Scene {si.get("sn","?")}: {si.get("title","")}</div><div style="font-size:14px;padding:2px 0;">{"█"*fl}{"░"*(30-fl)} {done[0]}/{total} ({p:.0%})</div><div style="color:#a0a0a0;font-size:12px;">⏱️ {el:.0f}s · clip: {ce:.0f}s · ~{eta:.0f}s left</div></div>'))
        except:pass
        tick+=1;_time.sleep(1)
for i,pd in enumerate(prompts):
    sn=pd['scene'];dur=pd.get('duration',model['ds'])
    si={'sn':sn,'title':pd['title'],'cs':_time.time()};done=[len(generated_videos)]
    _stop=False;t=threading.Thread(target=_timer,args=(prog,si,done,len(prompts),t0),daemon=True);t.start()
    try:
        inp={'prompt':pd['t2v_prompt']}
        if 'wan' in CHOSEN_MODEL:inp['aspect_ratio']='16:9';inp['enable_prompt_expansion']=False
        result=fal_client.subscribe(model['endpoint'],arguments=inp);url=None
        if isinstance(result,dict):
            if 'video' in result and isinstance(result['video'],dict):url=result['video'].get('url')
            elif 'video' in result:url=result['video']
            elif 'url' in result:url=result['url']
        if url:
            vp=VIDEOS_DIR/f'scene_{sn:03d}.mp4'
            with open(vp,'wb') as f:f.write(requests.get(url,timeout=120).content)
            generated_videos.append({'scene':sn,'path':str(vp),'duration':dur,'source':'generated'})
    except:pass
    finally:_stop=True;t.join(timeout=2)
el=_time.time()-t0
prog.update(HTML(f'<div style="background:#1a1a2e;padding:12px;border-radius:8px;color:#eaeaea;font-family:monospace;"><div style="font-size:16px;font-weight:bold;color:#4ecca3;">✅ {len(generated_videos)}/{len(prompts)} videos in {el:.0f}s</div></div>'))
with open(SESSION_DIR/'video_manifest.json','w') as f:json.dump({'model':CHOSEN_MODEL,'videos':generated_videos},f,indent=2)

In [ ]:
# PREVIEW
for v in sorted(generated_videos,key=lambda x:x['scene']):
    display(Markdown(f'### 🤖 Scene {v["scene"]}'))
    try:display(Video(v['path'],embed=True,width=640))
    except:print(v['path'])
    display(Markdown('---'))

---
*ROTBOTS Workshop — LI-MA 2026*